# Use Case #5: AI-Powered Customer Support with Intelligent Escalation

## How BigCompany Inc. Built a Smart Support System Using CrewAI Crews + Flows

**The #5 use case for CrewAI in 2026 — where Crews and Flows work together for maximum impact.**

## The Problem: BigCompany Inc. Is Drowning in Support Tickets

**BigCompany Inc.** is a fast-growing SaaS company that sells a project management platform called **BigProject Pro**. Their product is popular — but their customer support team has a serious problem.

Every day, hundreds of support tickets pour in. Some are simple questions that could be answered in seconds ("How do I reset my password?"). Others are complex issues that require careful investigation ("I was charged twice and need a refund"). But **all tickets go into the same queue**, and every ticket gets the same treatment:

- They receive **400+ tickets per day**.
- Support agents spend **15 minutes** on simple FAQ questions that should take 30 seconds.
- **Complex billing and technical issues** wait in queue behind password resets.
- Average first-response time has ballooned to **4.5 hours**.
- Customer satisfaction (CSAT) has dropped to **62%** — well below the industry average of 78%.
- The team is **burned out** from answering the same simple questions hundreds of times.

The Head of Customer Support calls an emergency meeting:

> *"We need a system that can automatically figure out what kind of ticket this is, answer the simple ones instantly, and route the complex ones to a full investigation team. Our agents should only deal with tickets that actually need a human brain."*

The engineering team proposes building an **AI-powered customer support system** using **CrewAI Flows + Crews**.

## Why CrewAI? And What's New Here — Flows!

In the previous notebooks, we learned how to build **Crews** — teams of AI agents that collaborate on tasks. A Crew is great for solving a single problem (like scoring a lead or writing content).

But customer support has a challenge that a single Crew can't handle well: **routing**.

Not every ticket needs the same treatment:
- A **simple question** ("What's your pricing?") should be answered by a single agent using the FAQ.
- A **complex issue** ("I was double-charged $450") should go to a full multi-agent investigation crew.

This is where **CrewAI Flows** come in.

### What Is a Flow?

A **Flow** is a structured workflow that orchestrates *when* and *how* things happen. Think of it as the **manager** that decides the sequence of operations:

| Concept | What It Does | Analogy |
|---------|-------------|---------|
| **Crew** | A team of agents collaborating on a shared task | A team of workers in a department |
| **Flow** | An orchestrator that routes work to the right crew | The floor manager who assigns tickets to the right department |

### Flow Decorators — The Key Concept

Flows use **Python decorators** — special annotations that define when and how each step executes:

| Decorator | What It Does | Example |
|-----------|-------------|---------|
| `@start()` | Marks the entry point of the flow | Receiving a new support ticket |
| `@listen(method)` | Runs when a specific method completes | After the ticket is classified |
| `@router(method)` | Routes to different paths based on a condition | Simple ticket → fast path; complex → full crew |
| `or_(a, b)` | Fires when **either** method a or b completes | After whichever path handled the ticket |

### The Big Picture: Crews Inside Flows

The real power comes from **combining Crews and Flows**:

```
Flow (the orchestrator)
  ├── @start()  →  Classify the ticket
  ├── @router() →  Route based on category
  │     ├── "simple"  →  Single-agent Crew (FAQ lookup)
  │     └── "complex" →  Multi-agent Crew (Research + Solution)
  └── @listen()  →  Format and deliver the response
```

This is CrewAI's **dual architecture** — Flows handle the routing logic, Crews handle the collaborative problem-solving.

## The Solution: How the AI Support System Works

Here's the workflow BigCompany Inc. designed:

```
New ticket arrives (e.g., "I can't reset my password")
        │
        ▼
┌─────────────────────┐
│  @start()           │  ◄── Step 1: Classify the ticket
│  classify_ticket()  │      (category + urgency)
└────────┬────────────┘
         │
         ▼
┌─────────────────────┐
│  @router()          │  ◄── Step 2: Route to the right path
│  route_ticket()     │      Returns "simple" or "complex"
└────────┬────────────┘
         │
    ┌────┴─────┐
    │          │
    ▼          ▼
┌────────┐ ┌──────────────┐
│ SIMPLE │ │   COMPLEX    │
│ path   │ │   path       │
│        │ │              │
│ 1 agent│ │ 2 agents:    │
│ + FAQ  │ │ Researcher + │
│ tool   │ │ Solution     │
│        │ │ Writer       │
└───┬────┘ └──────┬───────┘
    │             │
    └──────┬──────┘
           │
           ▼
┌─────────────────────┐
│  @listen(or_(...))  │  ◄── Step 3: Format final response
│  format_response()  │      (runs after either path)
└─────────────────────┘
```

The **Flow** handles the routing (Steps 1, 2, 3). The **Crews** handle the actual work (the simple path and the complex path).

## Let's Build It! Step by Step

### Step 0: Install the Required Packages

We need:
- **`crewai`** (v1.9.3) — the framework for building agents, crews, and flows.
- **`python-dotenv`** — to load our API key from a `.env` file.

In [ ]:
# Install packages (run this once)
#!pip install -U crewai python-dotenv

### Step 1: Load Environment Variables

Our agents need an **OpenAI API key** to use GPT models as their "brain." We store this securely in a `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("OpenAI API Key loaded successfully!")
else:
    print("WARNING: No OpenAI API Key found. Please set it in your .env file.")

### Step 2: Create BigCompany Inc.'s Support Database

In a real company, support data would come from a knowledge base like Zendesk or Confluence. For this demo, we'll simulate it with Python dictionaries.

We have two data sources:
- **FAQ_DB** — Common questions with pre-written answers (for the simple path).
- **KNOWLEDGE_BASE** — Detailed documentation about policies and procedures (for the complex path).

In [ ]:
# ============================================================
# BigCompany Inc.'s Support Knowledge Bases (simulated)
# ============================================================

# Simple FAQ — pre-written answers to common questions
FAQ_DB = {
    "password_reset": {
        "question": "How do I reset my password?",
        "answer": (
            "To reset your password:\n"
            "1. Go to app.bigproject.com/login\n"
            "2. Click 'Forgot Password'\n"
            "3. Enter your email address\n"
            "4. Check your inbox for a reset link (arrives within 2 minutes)\n"
            "5. Click the link and set a new password\n\n"
            "If you don't receive the email, check your spam folder."
        ),
    },
    "pricing": {
        "question": "What are your pricing plans?",
        "answer": (
            "BigProject Pro pricing:\n"
            "- Free: Up to 5 users, basic features\n"
            "- Team: $15/user/month — unlimited projects, integrations\n"
            "- Enterprise: $45/user/month — SSO, audit logs, priority support\n\n"
            "All paid plans include a 14-day free trial. "
            "Visit bigproject.com/pricing for details."
        ),
    },
    "cancel_subscription": {
        "question": "How do I cancel my subscription?",
        "answer": (
            "To cancel your subscription:\n"
            "1. Go to Settings > Billing\n"
            "2. Click 'Manage Subscription'\n"
            "3. Select 'Cancel Plan'\n"
            "4. Confirm cancellation\n\n"
            "Your access continues until the end of the current billing period. "
            "No refunds for partial months."
        ),
    },
    "integrations": {
        "question": "What integrations do you support?",
        "answer": (
            "BigProject Pro integrates with 200+ tools including:\n"
            "- Slack, Microsoft Teams, Discord\n"
            "- GitHub, GitLab, Bitbucket\n"
            "- Jira, Linear, Asana\n"
            "- Google Drive, Dropbox, OneDrive\n\n"
            "Visit Settings > Integrations to connect your tools."
        ),
    },
}

# Detailed knowledge base — for complex ticket investigation
KNOWLEDGE_BASE = {
    "billing": {
        "topic": "Billing & Refund Policies",
        "content": (
            "BILLING POLICIES:\n"
            "- Charges are processed on the 1st of each month.\n"
            "- Double charges can occur if a payment fails and retries.\n"
            "- Refund policy: Full refunds within 30 days of charge.\n"
            "- Partial refunds: Pro-rated for mid-cycle cancellations on annual plans.\n"
            "- To process a refund, a support agent must verify the charge in Stripe.\n"
            "- Refunds take 5-10 business days to appear on the customer's statement.\n"
            "- Escalate disputes over $500 to the billing manager."
        ),
    },
    "technical": {
        "topic": "Technical Troubleshooting",
        "content": (
            "COMMON TECHNICAL ISSUES:\n"
            "- API rate limits: 1000 requests/minute (Team), 5000/minute (Enterprise).\n"
            "- File upload limit: 100MB per file, 10GB total storage (Team).\n"
            "- 500 errors on upload: Usually caused by files exceeding size limit.\n"
            "- Sync issues: Clear browser cache, check internet connection.\n"
            "- SSO login failures: Verify SAML configuration in Settings > Security.\n"
            "- Webhook failures: Check endpoint URL and ensure HTTPS.\n"
            "- For persistent 500 errors, collect the request ID from the error page "
            "and escalate to the engineering team."
        ),
    },
    "account": {
        "topic": "Account Management",
        "content": (
            "ACCOUNT POLICIES:\n"
            "- Account deletion: Permanent, cannot be undone. 30-day grace period.\n"
            "- Data export: Available in Settings > Data > Export (CSV or JSON).\n"
            "- Team transfers: Admin can transfer ownership in Settings > Team.\n"
            "- Suspended accounts: Due to payment failure. Reactivate by updating payment.\n"
            "- GDPR data requests: Must be processed within 30 days."
        ),
    },
}

print("BigCompany Inc. support database loaded!")
print(f"  - {len(FAQ_DB)} FAQ entries (for simple tickets)")
print(f"  - {len(KNOWLEDGE_BASE)} knowledge base topics (for complex tickets)")

### Step 3: Define the Agent's Tools

**Tools** are functions that agents can call to look up information. We'll create two tools:

| Tool | What It Does | Used By |
|------|-------------|--------|
| `search_faq` | Searches the FAQ database for matching answers | FAQ Agent (simple path) |
| `search_knowledge_base` | Searches detailed documentation | Research Agent (complex path) |

The `@tool` decorator turns a regular Python function into a tool that agents can use. The **docstring** is critical — the LLM reads it to decide *when* and *how* to use each tool.

In [ ]:
from crewai.tools import tool


@tool("search_faq")
def search_faq(query: str) -> str:
    """Search the FAQ database for answers to common customer questions.

    Use this tool to find pre-written answers for simple questions about
    password resets, pricing, cancellations, and integrations.

    Args:
        query: A keyword or topic to search for (e.g., 'password', 'pricing').
    """
    query_lower = query.lower()
    results = []

    for key, entry in FAQ_DB.items():
        if (
            query_lower in key
            or query_lower in entry["question"].lower()
            or query_lower in entry["answer"].lower()
        ):
            results.append(f"Q: {entry['question']}\nA: {entry['answer']}")

    if results:
        return "\n\n---\n\n".join(results)
    return f"No FAQ entries found for '{query}'. Try different keywords."


@tool("search_knowledge_base")
def search_knowledge_base(topic: str) -> str:
    """Search the detailed knowledge base for policies and procedures.

    Use this tool to find in-depth information about billing policies,
    technical troubleshooting steps, or account management procedures.

    Args:
        topic: The topic to search for (e.g., 'billing', 'technical', 'account').
    """
    topic_lower = topic.lower()
    results = []

    for key, entry in KNOWLEDGE_BASE.items():
        if (
            topic_lower in key
            or topic_lower in entry["topic"].lower()
            or topic_lower in entry["content"].lower()
        ):
            results.append(f"Topic: {entry['topic']}\n{entry['content']}")

    if results:
        return "\n\n---\n\n".join(results)
    return f"No knowledge base entries found for '{topic}'. Try different keywords."


print("Tools created:")
print(f"  - search_faq: Searches {len(FAQ_DB)} FAQ entries")
print(f"  - search_knowledge_base: Searches {len(KNOWLEDGE_BASE)} KB topics")

### Step 4: Create the Agents

We need four agents — each with a specific role in the support pipeline:

| Agent | Role | Used In | Tools |
|-------|------|---------|-------|
| **Classifier** | Categorizes tickets by type and urgency | Flow (classification step) | None |
| **FAQ Specialist** | Answers simple questions from the FAQ | Simple path crew | `search_faq` |
| **Support Researcher** | Investigates complex issues | Complex path crew | `search_knowledge_base` |
| **Solution Writer** | Writes detailed responses | Complex path crew | None |

Notice how:
- The **Classifier** and **FAQ Specialist** work alone (single-agent crews).
- The **Researcher** and **Solution Writer** work together (multi-agent crew for complex issues).

#### Update note: current OpenAI models
* The original example is very good for learning the concept, and it worked with the OpenAI model it was recorded with: `gpt-4o-mini`, with `temperature=0`.
* OpenAI is retiring `gpt-4o-mini` (shutdown planned for January 14, 2027). With the current models, we would use `gpt-6-luna`, the small model OpenAI recommends now. `MODEL` defines it once, so you can change it in one line.
* Recent OpenAI models only accept the default temperature (1), so the code uses `temperature=1`. The idea behind `temperature=0` (more predictable answers, useful to classify tickets the same way every time) is still important, and other models still let you choose it.
* The agents of this flow use tools. Through the endpoint CrewAI uses, `gpt-6-luna` only accepts tools when its reasoning is turned off, so the code adds `extra_body={"reasoning_effort": "none"}`. The model then answers directly, like `gpt-4o-mini` did. It works with the version of CrewAI in the backend repository (1.9.3) and with the latest version that `pip install -U crewai` installs.
* Check available models at developers.openai.com/api/docs/deprecations.

In [ ]:
from crewai import Agent, LLM

# Configure the LLM (the "brain" for all agents)
# Original version: LLM(model="openai/gpt-4o-mini", temperature=0).
# OpenAI is retiring gpt-4o-mini (shutdown planned for January 14, 2027),
# so we use gpt-6-luna, the small model OpenAI recommends now.
# Check available models at developers.openai.com/api/docs/deprecations
MODEL = "openai/gpt-6-luna"

# Recent OpenAI models only accept the default temperature (1), so we use 1
# instead of 0. The agents of this flow use tools, and through the endpoint
# CrewAI uses (Chat Completions), gpt-6-luna only accepts tools when its
# reasoning is turned off, so we set reasoning_effort to "none". We send it
# with extra_body, which works with this project's version of CrewAI and
# with the latest one.
llm = LLM(model=MODEL, temperature=1, extra_body={"reasoning_effort": "none"})

# ── Agent 1: Ticket Classifier ─────────────────────────────
classifier_agent = Agent(
    role="Customer Support Ticket Classifier",
    goal=(
        "Accurately classify incoming support tickets by category "
        "and urgency level so they can be routed to the right team"
    ),
    backstory=(
        "You are a senior support operations specialist with 8 years "
        "of experience in customer support triage. You can quickly "
        "identify whether a ticket is a simple FAQ question or a "
        "complex issue requiring investigation. You never misclassify "
        "a billing dispute as a simple question."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# ── Agent 2: FAQ Specialist ─────────────────────────────────
faq_agent = Agent(
    role="FAQ Support Specialist",
    goal=(
        "Quickly answer common customer questions using the FAQ "
        "database, providing clear and helpful responses"
    ),
    backstory=(
        "You are a friendly and efficient frontline support agent. "
        "You know the FAQ inside and out and can answer common "
        "questions in a warm, helpful tone. You always point "
        "customers to the exact steps they need to follow."
    ),
    llm=llm,
    tools=[search_faq],
    verbose=True,
    allow_delegation=False,
)

# ── Agent 3: Support Researcher ─────────────────────────────
research_agent = Agent(
    role="Support Research Specialist",
    goal=(
        "Thoroughly investigate complex customer issues by searching "
        "the knowledge base for relevant policies and procedures"
    ),
    backstory=(
        "You are a detail-oriented support investigator who leaves "
        "no stone unturned. When a customer has a billing dispute "
        "or technical problem, you dig into the knowledge base to "
        "find every relevant policy, procedure, and troubleshooting "
        "step before passing your findings to the solution team."
    ),
    llm=llm,
    tools=[search_knowledge_base],
    verbose=True,
    allow_delegation=False,
)

# ── Agent 4: Solution Writer ────────────────────────────────
solution_agent = Agent(
    role="Customer Solution Specialist",
    goal=(
        "Write a clear, empathetic, and actionable response to the "
        "customer based on the research findings"
    ),
    backstory=(
        "You are a senior support agent known for turning frustrated "
        "customers into happy ones. You write responses that are "
        "empathetic, clear, and action-oriented. You always "
        "acknowledge the customer's frustration, explain what "
        "happened, and provide concrete next steps."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("Support team created!")
print(f"  - Agent 1: {classifier_agent.role}")
print(f"  - Agent 2: {faq_agent.role}")
print(f"  - Agent 3: {research_agent.role}")
print(f"  - Agent 4: {solution_agent.role}")

### Step 5: Define Structured Output Models

Just like in the previous use cases, we use **Pydantic models** to ensure our agents return data in a consistent, structured format.

We define two output "forms":
- **TicketClassification** — The classifier's output (category, complexity, urgency).
- **SupportResponse** — The final answer to the customer.

In [ ]:
from pydantic import BaseModel, Field
from typing import List


# ── Output format for the Classifier Agent ──────────────────
class TicketClassification(BaseModel):
    """Structured output from the Ticket Classifier."""

    category: str = Field(
        description=(
            "Ticket category: 'password_reset', 'pricing', "
            "'cancel_subscription', 'integrations', 'billing_issue', "
            "'technical_issue', or 'account_issue'"
        )
    )
    complexity: str = Field(
        description="'simple' (FAQ can answer) or 'complex' (needs investigation)"
    )
    urgency: str = Field(
        description="'low', 'medium', or 'high'"
    )
    reasoning: str = Field(
        description="Brief explanation of why this classification was chosen"
    )


# ── Output format for Support Response ──────────────────────
class SupportResponse(BaseModel):
    """Structured response to send to the customer."""

    subject: str = Field(description="Response subject line")
    message: str = Field(description="The full response message to the customer")
    follow_up_needed: bool = Field(
        description="Whether a human agent should follow up"
    )


print("Structured output models defined:")
print("  - TicketClassification: 4 fields (category, complexity, urgency, reasoning)")
print("  - SupportResponse: 3 fields (subject, message, follow_up_needed)")

### Step 6: Define the Flow State

A **Flow** needs a place to store data as it moves through each step. This is called the **Flow State**.

We define a Pydantic model that holds everything about the ticket as it gets processed:

```
Step 1 (classify): fills in → category, complexity, urgency
Step 2 (route):    reads   → complexity (to decide the path)
Step 3 (handle):   fills in → resolution
Step 4 (format):   reads   → everything, returns final output
```

The state is shared across all flow methods via `self.state`.

In [ ]:
class SupportTicketState(BaseModel):
    """State that flows through the entire support pipeline."""

    # Input fields (set before kickoff)
    customer_name: str = ""
    customer_email: str = ""
    ticket_subject: str = ""
    ticket_message: str = ""

    # Classification fields (set by classify_ticket)
    category: str = ""
    complexity: str = ""  # "simple" or "complex"
    urgency: str = ""

    # Resolution fields (set by handle_simple or handle_complex)
    resolution_subject: str = ""
    resolution_message: str = ""
    follow_up_needed: bool = False
    path_taken: str = ""  # "simple" or "complex" — for tracking


print("Flow state model defined: SupportTicketState")
print("  Input fields: customer_name, customer_email, ticket_subject, ticket_message")
print("  Classification fields: category, complexity, urgency")
print("  Resolution fields: resolution_subject, resolution_message, follow_up_needed")

### Step 7: Build the Flow — The Heart of the Application

Now we build the **CustomerSupportFlow** — the orchestrator that ties everything together.

This is where the new concepts come in:

1. **`@start()`** — The entry point. When we call `flow.kickoff()`, this method runs first. It creates a single-agent Crew to classify the ticket.

2. **`@router(classify_ticket)`** — After classification, this method examines `self.state.complexity` and returns either `"simple"` or `"complex"`. This string becomes a **route label**.

3. **`@listen("simple")`** — This method only runs if the router returned `"simple"`. It creates a single-agent FAQ Crew.

4. **`@listen("complex")`** — This method only runs if the router returned `"complex"`. It creates a multi-agent Crew with a Researcher and a Solution Writer.

5. **`@listen(or_(handle_simple_ticket, handle_complex_ticket))`** — This method runs after **either** path completes. `or_()` means "fire when any of these methods finish." It formats the final output.

Let's look at the code:

In [ ]:
from crewai import Task, Crew, Process
from crewai.flow.flow import Flow, start, listen, router, or_


class CustomerSupportFlow(Flow[SupportTicketState]):
    """
    BigCompany Inc.'s AI-powered customer support flow.

    This Flow orchestrates the entire support pipeline:
    1. Classify the ticket (category + complexity)
    2. Route to the right path (simple FAQ or complex investigation)
    3. Resolve the ticket using the appropriate Crew
    4. Format and deliver the final response
    """

    # ── Step 1: Classify the Ticket ───────────────────────────
    @start()
    def classify_ticket(self):
        """Entry point: classify the incoming ticket."""
        print(f"\n{'='*60}")
        print(f"NEW TICKET from {self.state.customer_name}")
        print(f"Subject: {self.state.ticket_subject}")
        print(f"{'='*60}")

        # Create a classification task
        classify_task = Task(
            description=(
                f"Classify this customer support ticket:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Subject: {self.state.ticket_subject}\n"
                f"  Message: {self.state.ticket_message}\n\n"
                f"Determine:\n"
                f"1. Category: password_reset, pricing, cancel_subscription, "
                f"integrations, billing_issue, technical_issue, or account_issue\n"
                f"2. Complexity: 'simple' if it's a common FAQ question, "
                f"'complex' if it requires investigation (billing disputes, "
                f"technical bugs, account problems)\n"
                f"3. Urgency: 'low', 'medium', or 'high'\n"
                f"   (billing disputes and data loss = high, "
                f"feature questions = low)"
            ),
            expected_output=(
                "A classification with category, complexity, urgency, "
                "and reasoning."
            ),
            agent=classifier_agent,
            output_pydantic=TicketClassification,
        )

        # Run a single-agent crew for classification
        classify_crew = Crew(
            agents=[classifier_agent],
            tasks=[classify_task],
            process=Process.sequential,
            verbose=True,
        )
        classify_crew.kickoff()

        # Store classification results in flow state
        if classify_task.output.pydantic:
            classification = classify_task.output.pydantic
            self.state.category = classification.category
            self.state.complexity = classification.complexity
            self.state.urgency = classification.urgency

        print(f"\nClassification: {self.state.category} | "
              f"{self.state.complexity} | urgency={self.state.urgency}")

        return self.state.complexity

    # ── Step 2: Route the Ticket ──────────────────────────────
    @router(classify_ticket)
    def route_ticket(self):
        """
        Router: decides which path the ticket takes.

        Returns a string — either "simple" or "complex".
        Methods decorated with @listen("simple") or @listen("complex")
        will fire based on this return value.
        """
        if self.state.complexity == "simple":
            print("\n>> ROUTING: Simple path (FAQ agent)")
            return "simple"
        else:
            print("\n>> ROUTING: Complex path (investigation crew)")
            return "complex"

    # ── Step 3a: Handle Simple Tickets (FAQ Path) ─────────────
    @listen("simple")
    def handle_simple_ticket(self):
        """
        Runs only when route_ticket() returns "simple".
        Uses a single FAQ agent to answer the question.
        """
        print("\n--- Simple Path: FAQ Agent handling ticket ---")

        faq_task = Task(
            description=(
                f"Answer this customer support question using the FAQ database:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Question: {self.state.ticket_message}\n\n"
                f"Search the FAQ for relevant information and write a friendly, "
                f"helpful response. Address the customer by name."
            ),
            expected_output="A helpful response with clear instructions.",
            agent=faq_agent,
            output_pydantic=SupportResponse,
        )

        faq_crew = Crew(
            agents=[faq_agent],
            tasks=[faq_task],
            process=Process.sequential,
            verbose=True,
        )
        faq_crew.kickoff()

        # Store resolution in flow state
        if faq_task.output.pydantic:
            response = faq_task.output.pydantic
            self.state.resolution_subject = response.subject
            self.state.resolution_message = response.message
            self.state.follow_up_needed = response.follow_up_needed

        self.state.path_taken = "simple"
        return self.state.resolution_message

    # ── Step 3b: Handle Complex Tickets (Investigation Path) ──
    @listen("complex")
    def handle_complex_ticket(self):
        """
        Runs only when route_ticket() returns "complex".
        Uses a multi-agent crew: Researcher + Solution Writer.
        """
        print("\n--- Complex Path: Investigation Crew handling ticket ---")

        # Task 1: Research the issue
        research_task = Task(
            description=(
                f"Investigate this customer support issue:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Category: {self.state.category}\n"
                f"  Issue: {self.state.ticket_message}\n\n"
                f"Search the knowledge base for relevant policies, procedures, "
                f"and troubleshooting steps. Compile all relevant findings."
            ),
            expected_output=(
                "A detailed research report with relevant policies, "
                "procedures, and recommended actions."
            ),
            agent=research_agent,
        )

        # Task 2: Write the solution response
        solution_task = Task(
            description=(
                f"Based on the research findings, write a response to "
                f"{self.state.customer_name} about their issue:\n"
                f"  Issue: {self.state.ticket_message}\n\n"
                f"Guidelines:\n"
                f"- Acknowledge their frustration\n"
                f"- Explain what happened (if applicable)\n"
                f"- Provide clear next steps\n"
                f"- Be empathetic and professional\n"
                f"- Set follow_up_needed=True if a human should follow up"
            ),
            expected_output=(
                "A professional, empathetic customer response with "
                "clear next steps."
            ),
            agent=solution_agent,
            context=[research_task],
            output_pydantic=SupportResponse,
        )

        # Run the multi-agent investigation crew
        investigation_crew = Crew(
            agents=[research_agent, solution_agent],
            tasks=[research_task, solution_task],
            process=Process.sequential,
            verbose=True,
        )
        investigation_crew.kickoff()

        # Store resolution in flow state
        if solution_task.output.pydantic:
            response = solution_task.output.pydantic
            self.state.resolution_subject = response.subject
            self.state.resolution_message = response.message
            self.state.follow_up_needed = response.follow_up_needed

        self.state.path_taken = "complex"
        return self.state.resolution_message

    # ── Step 4: Format Final Response ─────────────────────────
    @listen(or_(handle_simple_ticket, handle_complex_ticket))
    def format_final_response(self, resolution):
        """
        Runs after EITHER the simple or complex path completes.

        or_() means: fire when ANY of these methods finish.
        Since only one path runs per ticket (thanks to the router),
        this effectively means "run after whichever path was taken."
        """
        print(f"\n--- Formatting final response (path: {self.state.path_taken}) ---")

        # Build the final formatted response
        final = (
            f"To: {self.state.customer_name} <{self.state.customer_email}>\n"
            f"Subject: {self.state.resolution_subject}\n"
            f"Path: {self.state.path_taken.upper()} | "
            f"Category: {self.state.category} | "
            f"Urgency: {self.state.urgency}\n"
            f"Follow-up needed: {'Yes' if self.state.follow_up_needed else 'No'}\n"
            f"{'='*60}\n\n"
            f"{self.state.resolution_message}"
        )

        return final


print("CustomerSupportFlow defined!")
print("  Steps:")
print("    1. @start()              → classify_ticket")
print("    2. @router()             → route_ticket (returns 'simple' or 'complex')")
print("    3a. @listen('simple')    → handle_simple_ticket (FAQ crew)")
print("    3b. @listen('complex')   → handle_complex_ticket (investigation crew)")
print("    4. @listen(or_(...))     → format_final_response")

## Let's Test Our AI Support System!

Now comes the fun part — let's process real support tickets through the system.

We'll test three scenarios:
1. **A simple ticket** — "How do I reset my password?" (should take the FAQ path)
2. **A complex ticket** — "I was charged twice" (should take the investigation path)
3. **Another complex ticket** — "API keeps returning errors" (technical investigation)

For each test, we:
1. Create a new `CustomerSupportFlow` instance.
2. Set the ticket data on `flow.state`.
3. Call `flow.kickoff()` to start the pipeline.
4. Examine the results.

### Test 1: A Simple Ticket — "How do I reset my password?"

This is a textbook FAQ question. The system should:
1. Classify it as `password_reset` / `simple` / `low` urgency.
2. Route it to the **simple path** (FAQ agent).
3. Answer from the FAQ database.
4. No follow-up needed.

In [ ]:
# Create a new flow instance for this ticket
flow_1 = CustomerSupportFlow()

# Set the ticket data
flow_1.state.customer_name = "Alice Johnson"
flow_1.state.customer_email = "alice.johnson@example.com"
flow_1.state.ticket_subject = "Can't log in — need password reset"
flow_1.state.ticket_message = (
    "Hi, I forgot my password and I can't log into my account. "
    "How do I reset it? Thanks!"
)

# Run the flow
result_1 = flow_1.kickoff()

print("\nFlow execution complete!")

In [ ]:
# Display the final response
print("=" * 60)
print("FINAL RESPONSE — Test 1 (Simple Ticket)")
print("=" * 60)
print(result_1)

Notice how the system correctly:
1. **Classified** the ticket as a simple FAQ question.
2. **Routed** it to the FAQ path (not the investigation crew).
3. **Answered** using the FAQ database with clear step-by-step instructions.
4. Marked **no follow-up needed** — this was handled fully by the AI.

A human agent didn't need to touch this ticket at all!

### Test 2: A Complex Ticket — "I was charged twice!"

This is a billing dispute — it requires investigation. The system should:
1. Classify it as `billing_issue` / `complex` / `high` urgency.
2. Route it to the **complex path** (investigation crew).
3. Research billing policies in the knowledge base.
4. Write an empathetic, actionable response.
5. Flag for human follow-up.

In [ ]:
# Create a new flow instance for this ticket
flow_2 = CustomerSupportFlow()

# Set the ticket data
flow_2.state.customer_name = "Robert Martinez"
flow_2.state.customer_email = "r.martinez@techstartup.io"
flow_2.state.ticket_subject = "URGENT — Charged twice this month!"
flow_2.state.ticket_message = (
    "I just checked my bank statement and I was charged $45 TWICE "
    "for my Enterprise subscription this month! I need one of these "
    "charges refunded immediately. This is really frustrating — "
    "I've been a loyal customer for 2 years."
)

# Run the flow
result_2 = flow_2.kickoff()

print("\nFlow execution complete!")

In [ ]:
# Display the final response
print("=" * 60)
print("FINAL RESPONSE — Test 2 (Complex Ticket: Billing Dispute)")
print("=" * 60)
print(result_2)

Notice the difference from Test 1:
1. The ticket was **classified as complex** (billing dispute, high urgency).
2. It was **routed to the investigation crew** — two agents worked on it.
3. The **Research Agent** searched the knowledge base and found billing policies.
4. The **Solution Writer** crafted an empathetic, detailed response.
5. **Follow-up is needed** — a human agent should verify and process the refund.

The AI handled the research and drafted the response, but the actual refund still requires a human. This is the **intelligent escalation** pattern.

### Test 3: A Complex Ticket — "API keeps returning 500 errors"

This is a technical issue. Let's see how the system handles it.

In [ ]:
# Create a new flow instance for this ticket
flow_3 = CustomerSupportFlow()

# Set the ticket data
flow_3.state.customer_name = "Priya Sharma"
flow_3.state.customer_email = "priya@devagency.co"
flow_3.state.ticket_subject = "API 500 errors on file upload"
flow_3.state.ticket_message = (
    "We're getting 500 errors every time we try to upload files "
    "through the API. This started yesterday. Our files are around "
    "150MB each. This is blocking our entire workflow — we need "
    "this fixed ASAP."
)

# Run the flow
result_3 = flow_3.kickoff()

print("\nFlow execution complete!")

In [ ]:
# Display the final response
print("=" * 60)
print("FINAL RESPONSE — Test 3 (Complex Ticket: Technical Issue)")
print("=" * 60)
print(result_3)

The Research Agent found that the **100MB file upload limit** is likely the cause — the customer is uploading 150MB files. The Solution Writer explained the issue, suggested a workaround, and flagged for engineering follow-up if needed.

## Let's Compare the Three Paths

Let's see a summary of how all three tickets were handled:

In [ ]:
# Summary comparison
tickets = [
    ("Alice Johnson", flow_1),
    ("Robert Martinez", flow_2),
    ("Priya Sharma", flow_3),
]

print(f"{'Customer':<20} {'Category':<22} {'Path':<10} {'Urgency':<10} {'Follow-up'}")
print("-" * 80)

for name, flow in tickets:
    s = flow.state
    print(
        f"{name:<20} {s.category:<22} {s.path_taken:<10} "
        f"{s.urgency:<10} {'Yes' if s.follow_up_needed else 'No'}"
    )

## Recap: What We Built and How It Works

### The Architecture — Crews Inside a Flow

```
┌──────────────────────────────────────────────────────────────┐
│                    CustomerSupportFlow                       │
│                   (Flow[SupportTicketState])                 │
│                                                              │
│  ┌──────────────────────────────────────────────────────┐    │
│  │  @start() classify_ticket                            │    │
│  │  → Single-agent Crew: Classifier Agent               │    │
│  │  → Output: TicketClassification (category, complexity)    │
│  └────────────────────┬─────────────────────────────────┘    │
│                       │                                      │
│  ┌────────────────────▼─────────────────────────────────┐    │
│  │  @router() route_ticket                              │    │
│  │  → Returns "simple" or "complex"                     │    │
│  └───────┬───────────────────────────┬──────────────────┘    │
│          │                           │                       │
│  ┌───────▼──────────┐    ┌───────────▼──────────────┐        │
│  │ @listen("simple")     │ @listen("complex")       │        │
│  │                  │    │                          │        │
│  │ Single-agent     │    │ Multi-agent Crew:        │        │
│  │ Crew:            │    │  - Research Agent        │        │
│  │  - FAQ Agent     │    │  - Solution Writer       │        │
│  │  - search_faq    │    │  - search_knowledge_base │        │
│  │    tool          │    │    tool                  │        │
│  └───────┬──────────┘    └───────────┬──────────────┘        │
│          │                           │                       │
│  ┌───────▼───────────────────────────▼──────────────────┐    │
│  │  @listen(or_(...)) format_final_response             │    │
│  │  → Formats and delivers the response                 │    │
│  └──────────────────────────────────────────────────────┘    │
└──────────────────────────────────────────────────────────────┘
```

### Key Components

| Component | What It Is | In Our Code |
|-----------|-----------|-------------|
| **Flow** | The orchestrator that routes tickets | `CustomerSupportFlow` |
| **Flow State** | Shared data across all steps | `SupportTicketState` (Pydantic model) |
| **`@start()`** | Entry point of the flow | `classify_ticket()` |
| **`@router()`** | Conditional branching | `route_ticket()` → returns `"simple"` or `"complex"` |
| **`@listen("route")`** | Listens for a specific route | `handle_simple_ticket()`, `handle_complex_ticket()` |
| **`or_()`** | Fires when any method completes | `format_final_response()` runs after either path |
| **Crew** | Team of agents for a specific task | FAQ crew (1 agent), Investigation crew (2 agents) |
| **Agent** | An AI team member with a role | Classifier, FAQ Specialist, Researcher, Solution Writer |
| **Task** | A specific assignment for an agent | Classification, FAQ lookup, Research, Solution writing |
| **Tool** | A function agents can call | `search_faq`, `search_knowledge_base` |

## How BigCompany Inc. Benefits

After deploying this AI support system, here's what changed:

| Metric | Before | After |
|--------|--------|-------|
| Average first-response time | 4.5 hours | **Under 1 minute** (simple), **5 minutes** (complex) |
| Simple tickets handled by humans | 100% | **0%** (fully automated) |
| Complex tickets with research attached | 0% | **100%** (AI pre-researches every issue) |
| Customer satisfaction (CSAT) | 62% | **84%** |
| Support agent burnout | High | **Low** (agents focus on meaningful work) |
| Tickets processed per day | 100 (human limit) | **400+** (all of them) |

The key insight: **not every ticket needs the same level of effort**. The Flow's routing ensures:
- **Simple questions** get instant answers (no human needed).
- **Complex issues** get thorough investigation (AI does the research, human makes the decision).
- **No ticket falls through the cracks** — every ticket is classified, routed, and handled.

## Key Takeaways for Beginners

1. **Flows are the orchestrator; Crews are the workers.** A Flow decides *what* needs to happen and *when*. A Crew decides *how* to do it. Together, they create powerful, structured AI applications.

2. **`@start()` is the entry point.** When you call `flow.kickoff()`, all methods decorated with `@start()` execute first.

3. **`@router()` enables intelligent routing.** The router method returns a string (like `"simple"` or `"complex"`), and `@listen()` methods match on that string. This is how you create different paths for different situations.

4. **`@listen()` can listen to methods or route names.** `@listen(classify_ticket)` fires when that method completes. `@listen("simple")` fires when a router returns that string.

5. **`or_()` combines multiple triggers.** `@listen(or_(method_a, method_b))` fires when *either* method completes — perfect for merging parallel paths back together.

6. **Flow State shares data between steps.** The `SupportTicketState` Pydantic model flows through every step. Each method can read and write to `self.state`.

7. **You can call `crew.kickoff()` inside any flow method.** This is how you combine the routing power of Flows with the collaborative intelligence of Crews.

8. **Each test gets a fresh flow instance.** We create a new `CustomerSupportFlow()` for each ticket. The flow state is not shared between tickets.

## What's Next?

This is a simple but functional customer support system. In a production environment at BigCompany Inc., the team would add:

- **`@human_feedback()` decorator** — CrewAI's built-in mechanism for human-in-the-loop workflows. When the AI is uncertain, the flow can pause and present the drafted response to a human agent for approval before sending it to the customer. Import via `from crewai.flow.human_feedback import human_feedback`.

- **`@persist` decorator** — Saves the flow state to a SQLite database after each step. If the system restarts or a human agent takes hours to review an escalation, the workflow picks up exactly where it left off. Import via `from crewai.flow.persistence import persist`.

- **Real knowledge base integration via MCP** — Use CrewAI's Model Context Protocol (MCP) support to connect directly to Zendesk, Confluence, or other knowledge management systems instead of dictionaries.

- **More routing paths** — Add a `"critical"` route for data loss or security issues that bypasses the AI entirely and goes straight to a human.

- **Customer history** — Give agents access to the customer's previous tickets and account information for more personalized responses.

- **`and_()` for parallel processing** — Use `and_()` to wait for multiple agents to finish before proceeding. For example, run sentiment analysis and classification in parallel, then route based on both results.

But the core architecture — a **Flow** that classifies, routes, and orchestrates **Crews** of specialized agents — remains exactly the same.